# Aula 2 — Notebook 2
## Orquestração multiagente com LangGraph

Visão geral: neste notebook, vamos expandir no trabalho que já fizemos para decompor a investigação em agentes especializados usando LangGraph. Cada agente tem um papel claro e um conjunto de ferramentas (tools) que pode invocar quando precisar de evidências.

Objetivos do notebook:
- Entender como orquestrar vários agentes com responsabilidades separadas;
- Ver como padronizar tools para leitura/inspeção e conectá‑las a agentes LangGraph;
- Avaliar vantagens e custos da decomposição em múltiplos agentes (rastreabilidade vs. complexidade).

Fluxo esperado (alto nível):
```text
classificador → investigador → mapeador de módulos → propositor → revisor
```

Como este notebook funciona (passo a passo):
1. Inicializamos paths e índices TF-IDF (células de setup).
2. Transformamos funções utilitárias em tools compatíveis com LangGraph (wrap com `@tool`).
3. Criamos agentes com prompts específicos (papéis) que recebem as mesmas tools.
4. Encadeamos a execução: cada agente recebe o resumo do anterior e pode usar tools para buscar evidências.
5. Consolidamos e salvamos o artefato final para revisão humana.

Dica de execução: rode as células sequencialmente. Se alguma asserção de diretório falhar, crie a pasta `outputs/` no nível do projeto.

In [1]:
from pathlib import Path
import os
import json
import textwrap
import subprocess
import sys
from datetime import datetime

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
TARGET_PROJECT_DIR = PROJECT_ROOT / "target_project"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

for directory in [DATA_DIR, TARGET_PROJECT_DIR, OUTPUT_DIR]:
    assert directory.exists(), f"Diretório não encontrado: {directory}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("TARGET_PROJECT_DIR:", TARGET_PROJECT_DIR)

PROJECT_ROOT: c:\Users\cstefano\Desktop\Carlos\Alura\Skills and Go\Aulas\Aula 2_final_final\aula_2_agentes_tools_reais_target_project
DATA_DIR: c:\Users\cstefano\Desktop\Carlos\Alura\Skills and Go\Aulas\Aula 2_final_final\aula_2_agentes_tools_reais_target_project\data
TARGET_PROJECT_DIR: c:\Users\cstefano\Desktop\Carlos\Alura\Skills and Go\Aulas\Aula 2_final_final\aula_2_agentes_tools_reais_target_project\target_project


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXTENSOES_INDEXADAS = {".md", ".txt", ".log", ".py", ".json", ".toml"}


def ler_texto(caminho):
    return Path(caminho).read_text(encoding="utf-8")


def listar_arquivos_recursivamente(raiz, extensoes=EXTENSOES_INDEXADAS):
    arquivos = []
    for caminho in Path(raiz).rglob("*"):
        if caminho.is_file() and caminho.suffix in extensoes:
            arquivos.append(caminho)
    return sorted(arquivos)


def montar_corpus(data_dir=DATA_DIR, target_project_dir=TARGET_PROJECT_DIR):
    arquivos = listar_arquivos_recursivamente(data_dir) + listar_arquivos_recursivamente(target_project_dir)
    documentos = []
    for caminho in arquivos:
        relativo = caminho.relative_to(PROJECT_ROOT).as_posix()
        documentos.append({
            "path": relativo,
            "content": ler_texto(caminho),
            "source_type": "target_project" if "target_project" in relativo else "evidence",
        })
    return documentos


def criar_indice_tfidf(documentos):
    vectorizer = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2))
    matriz = vectorizer.fit_transform([doc["content"] for doc in documentos])
    return {"vectorizer": vectorizer, "matrix": matriz, "documents": documentos}


def buscar_no_indice(indice, consulta, k=5):
    consulta_vetor = indice["vectorizer"].transform([consulta])
    scores = cosine_similarity(consulta_vetor, indice["matrix"])[0]
    ordenados = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:k]
    resultados = []
    for idx, score in ordenados:
        doc = indice["documents"][idx]
        resultados.append({
            "path": doc["path"],
            "score": round(float(score), 4),
            "source_type": doc["source_type"],
            "preview": doc["content"][:1200],
        })
    return resultados


def listar_tickets():
    return [p.relative_to(PROJECT_ROOT).as_posix() for p in sorted((DATA_DIR / "tickets").glob("*.md"))]


def carregar_arquivo_relativo(caminho_relativo):
    caminho = PROJECT_ROOT / caminho_relativo
    if not caminho.exists():
        return json.dumps({"erro": f"Arquivo não encontrado: {caminho_relativo}"}, ensure_ascii=False)
    return caminho.read_text(encoding="utf-8")


def listar_arquivos_do_target_project():
    arquivos = listar_arquivos_recursivamente(TARGET_PROJECT_DIR)
    return [p.relative_to(PROJECT_ROOT).as_posix() for p in arquivos]


def executar_testes_target_project():
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=TARGET_PROJECT_DIR,
        capture_output=True,
        text=True,
        timeout=30,
    )
    return {
        "returncode": resultado.returncode,
        "stdout": resultado.stdout,
        "stderr": resultado.stderr,
    }


def salvar_json_saida(nome_arquivo, conteudo):
    caminho = OUTPUT_DIR / nome_arquivo
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(json.dumps(conteudo, ensure_ascii=False, indent=2), encoding="utf-8")
    return caminho.relative_to(PROJECT_ROOT).as_posix()

DOCUMENTOS = montar_corpus()
INDICE = criar_indice_tfidf(DOCUMENTOS)
print(f"Documentos indexados: {len(DOCUMENTOS)}")

Documentos indexados: 24


## Tools LangGraph

Objetivo desta seção: adaptar funções utilitárias (lista de arquivos, leitura, busca TF-IDF, execução de testes) para o formato de tools que LangGraph/LangChain entende.

Pontos importantes sobre as tools:
- Cada tool é uma função Python decorada com `@tool` (do LangChain). Isso transforma a função em um ponto de entrada que o agente pode invocar;
- As tools retornam strings JSON (ou texto) que são fáceis de serializar e inspecionar;
- As tools aqui são *leitura/inspeção* — não há tool que escreva ou modifique arquivos;
- Erros e exceções devem ser capturados pela implementação da tool para evitar que o agente receba stack traces cru.

Como as agents usam essas tools:
- O agente recebe as ferramentas no momento de sua criação (`create_react_agent(...)`) e pode chamá‑las quando necessário;
- As chamadas são síncronas: o agente invoca a tool, recebe o resultado, e continua a geração com essa evidência;
- O desenvolvedor controla quais tools cada agente vê, permitindo especialização e limites de autonomia.

Dica pedagógica: peça aos alunos para inspecionar o retorno de `buscar_evidencias(...)` e comparar os `preview`s com os arquivos originais usando `ler_arquivo(...)` para entender a cadeia de evidência.

In [25]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent


def exigir_openai_api_key():
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Defina OPENAI_API_KEY para executar os agentes reais.")


@tool
def buscar_evidencias(consulta: str, k: int = 5) -> str:
    """Busca evidências em tickets, logs, runbooks, histórico, políticas e código do target_project."""
    return json.dumps(buscar_no_indice(INDICE, consulta, k), ensure_ascii=False, indent=2)


@tool
def ler_arquivo(caminho_relativo: str) -> str:
    """Lê um arquivo pelo caminho relativo ao projeto do curso."""
    return carregar_arquivo_relativo(caminho_relativo)


@tool
def listar_tickets_disponiveis() -> str:
    """Lista tickets disponíveis para investigação."""
    return json.dumps(listar_tickets(), ensure_ascii=False, indent=2)


@tool
def listar_arquivos_codigo() -> str:
    """Lista arquivos disponíveis no target_project."""
    return json.dumps(listar_arquivos_do_target_project(), ensure_ascii=False, indent=2)


@tool
def executar_testes() -> str:
    """Executa os testes do target_project e retorna stdout, stderr e código de saída."""
    return json.dumps(executar_testes_target_project(), ensure_ascii=False, indent=2)


TOOLS_GERAIS = [
    buscar_evidencias,
    ler_arquivo,
    listar_tickets_disponiveis,
    listar_arquivos_codigo,
    executar_testes,
]


def criar_agente_langgraph(nome, papel, tools=TOOLS_GERAIS, modelo="gpt-4.1-mini"):
    exigir_openai_api_key()
    llm = ChatOpenAI(model=modelo, temperature=0)
    prompt = f"""
Você é {nome}.

Papel:
{papel}

Use as tools quando precisar consultar tickets, logs, runbooks, histórico, políticas,
arquivos do target_project ou testes. Não assuma evidências sem consultar fontes.
Não altere arquivos. Produza uma resposta objetiva e rastreável.
"""
    return create_agent(llm, tools=tools, system_prompt=prompt)


def executar_agente_langgraph(agente, mensagem):
    """Invoca o agente de forma resiliente, suportando diferentes interfaces retornadas por
    `create_agent` / executors antigos. Retorna texto bruto como string.
    """
    try:
        # Prefer explicit invoke with messages (newer executors)
        if hasattr(agente, "invoke"):
            resposta = agente.invoke({"messages": [{"role": "user", "content": mensagem}]})
            # resposta pode ser dict, objeto ou outro. Tentar extrair texto de forma defensiva.
            if isinstance(resposta, dict):
                if "messages" in resposta and isinstance(resposta["messages"], list) and resposta["messages"]:
                    last = resposta["messages"][-1]
                    # last pode ser dict ou objeto
                    if isinstance(last, dict):
                        return last.get("content") or last.get("text") or json.dumps(last, ensure_ascii=False)
                    return getattr(last, "content", getattr(last, "text", str(last)))
                if "output_text" in resposta:
                    return resposta["output_text"]
                if "output" in resposta:
                    return resposta["output"]
                return json.dumps(resposta, ensure_ascii=False)
            # resposta pode ser um objeto com atributos
            if hasattr(resposta, "output_text"):
                return resposta.output_text
            if hasattr(resposta, "content"):
                return resposta.content
            return str(resposta)

        # fallback: run(mensagem)
        if hasattr(agente, "run"):
            out = agente.run(mensagem)
            return out if isinstance(out, str) else json.dumps(out, ensure_ascii=False)

        # fallback: callable
        if callable(agente):
            out = agente(mensagem)
            return out if isinstance(out, str) else json.dumps(out, ensure_ascii=False)

        return f"Tipo de agente não suportado: {type(agente)}"
    except Exception as e:
        return f"Erro ao executar agente: {e}"


## Criando agentes especializados

Aqui definimos cinco agentes com papéis distintos (classificador, investigador, mapeador, propositor e revisor). Cada agente recebe as mesmas tools, mas seus prompts direcionam o comportamento.

Design de prompts e responsabilidades:
- `classificador`: objetivo é avaliar severidade e indicar quais evidências procurar; saída esperada: breve classificação e lista de investigações recomendadas;
- `investigador`: busca evidências e formula hipóteses técnicas, citando caminhos/trechos; saída esperada: hipóteses com evidências;
- `mapeador`: relaciona hipóteses a arquivos do `target_project`, sugerindo módulos afetados;
- `propositor`: traduz hipótese em proposta conceitual e plano de testes (sem alterar código);
- `revisor`: valida se a proposta é consistente com evidências e regras de autonomia, bloqueando se necessário.

Boas práticas ao construir agentes multicamadas:
- Defina claramente as entradas e saídas esperadas de cada agente;
- Instrua os agentes a sempre citar evidências (caminhos / trechos) em cada hipótese;
- Mantenha temperatura baixa (`temperature=0`) para resultados determinísticos durante aulas e experimentos controlados.

Nota para instrutores: peça que os alunos experimentem trocar ferramentas entre agentes (por exemplo, negar `executar_testes` ao `classificador`) para explorar impacto na investigação.

In [26]:
classificador = criar_agente_langgraph(
    "agente classificador de incidentes",
    "Classificar o incidente, identificar severidade inicial e apontar quais evidências precisam ser consultadas."
)

investigador = criar_agente_langgraph(
    "agente investigador técnico",
    "Consultar evidências externas e formular hipóteses técnicas sustentadas por arquivos, logs, runbooks ou histórico."
)

mapeador = criar_agente_langgraph(
    "agente mapeador de módulos afetados",
    "Relacionar as hipóteses técnicas aos arquivos do target_project que provavelmente precisam ser revisados."
)

propositor = criar_agente_langgraph(
    "agente propositor de correção conceitual",
    "Propor uma mudança conceitual e um plano mínimo de testes sem alterar código automaticamente."
)

revisor = criar_agente_langgraph(
    "agente revisor técnico e de limites de autonomia",
    "Revisar se a proposta está sustentada por evidências e se respeita limites de autonomia. Bloquear quando necessário."
)

## Execução encadeada

Descrição: o encadeamento é explícito: cada agente recebe o resumo textual produzido pelo agente anterior e pode (se necessário) invocar tools para obter evidências adicionais.

Como interpretar o encadeamento:
- O `classificador` produz uma classificação inicial (texto curto). Essa saída é passada como contexto ao `investigador`;
- O `investigador` pode chamar `buscar_evidencias` e `ler_arquivo` para sustentar hipóteses e passa um resumo para o `mapeador`;
- O `mapeador` mapeia hipóteses para arquivos concretos e retorna uma lista de módulos/arquivos para o `propositor`;
- O `propositor` cria uma proposta conceitual e um plano de testes; o `revisor` faz a validação final.

Dicas de depuração e inspeção:
- Imprima cada `etapa_n` (as variáveis de saída) para inspecionar o conteúdo intermediário;
- Quando o agente chama uma tool, compare o retorno com o arquivo original usando `ler_arquivo` para confirmar a fidelidade do preview;
- Se algo parecer vago, reexecute o agente com instruções mais estritas pedindo citações de linhas ou trechos específicos.

In [27]:
incident_id = "INC-001"

etapa_1 = executar_agente_langgraph(
    classificador,
    f"Classifique o incidente {incident_id}. Consulte o ticket e indique quais evidências devem ser buscadas."
)
print("ETAPA 1 — CLASSIFICAÇÃO")
print(etapa_1)

ETAPA 1 — CLASSIFICAÇÃO
Incidente INC-001:

- Classificação: Falha na ingestão de pedidos no sistema mini_orders_pipeline.
- Severidade inicial: Alta.
- Descrição: O job `orders_daily_ingestion` falhou na etapa de ingestão de pedidos devido a um erro `KeyError: 'customer_id'` após a entrada de um novo lote do parceiro `marketplace_beta`. Isso impactou o processamento dos pedidos e o dashboard diário de receita ficou incompleto.

Evidências que devem ser consultadas para investigação:
1. Logs do job `orders_daily_ingestion` para identificar o momento e contexto do erro.
2. Dados do lote enviado pelo parceiro `marketplace_beta` para verificar a presença e formato do campo `customer_id`.
3. Código e testes relacionados à validação do schema dos pedidos, especialmente a parte que trata do campo `customer_id`.
4. Histórico de alterações recentes no pipeline de ingestão e transformação de pedidos.
5. Tickets anteriores relacionados a erros de schema ou ingestão de pedidos.

Posso ajudar a bu

In [28]:
etapa_2 = executar_agente_langgraph(
    investigador,
    f"""
Investigue o incidente {incident_id} usando a classificação abaixo.

CLASSIFICAÇÃO:
{etapa_1}

Formule hipóteses técnicas sustentadas por evidências consultadas via tools.
"""
)
print("ETAPA 2 — INVESTIGAÇÃO")
print(etapa_2)

ETAPA 2 — INVESTIGAÇÃO
Sim, por favor, inicie a investigação buscando as evidências na seguinte ordem para melhor entendimento do incidente INC-001:

1. Logs do job `orders_daily_ingestion` para identificar o momento e contexto do erro.
2. Dados do lote enviado pelo parceiro `marketplace_beta`.
3. Código e testes relacionados à validação do schema dos pedidos, focando no campo `customer_id`.
4. Histórico de alterações recentes no pipeline de ingestão e transformação de pedidos.
5. Tickets anteriores relacionados a erros de schema ou ingestão de pedidos.

Solicito que busque as evidências 1 e 2 em paralelo para agilizar a investigação.


In [29]:
etapa_3 = executar_agente_langgraph(
    mapeador,
    f"""
Mapeie módulos afetados no target_project para o incidente {incident_id}.

INVESTIGAÇÃO:
{etapa_2}

Use tools para listar e ler arquivos relevantes do target_project.
"""
)
print("ETAPA 3 — MÓDULOS AFETADOS")
print(etapa_3)

ETAPA 3 — MÓDULOS AFETADOS
Para o incidente INC-001, que envolve falha na ingestão de pedidos devido a erro `KeyError: 'customer_id'` no job `orders_daily_ingestion` com lote do parceiro `marketplace_beta`, os módulos do target_project que provavelmente precisam ser revisados são:

1. **src/mini_orders_pipeline/validacao_schema.py**  
   - Contém a função `validar_pedido` que acessa diretamente o campo `customer_id` sem tratamento para ausência, o que pode causar `KeyError`.
   - A validação atual não gera erro estruturado com contexto, apenas levanta exceção genérica.

2. **src/mini_orders_pipeline/ingestao_pedidos.py**  
   - Orquestra o processamento do lote, chamando a validação e transformação.
   - Pode precisar de tratamento para capturar erros de validação e registrar contexto do erro.

3. **src/mini_orders_pipeline/transformacao_pedidos.py**  
   - Embora o erro seja na validação, a transformação assume que o campo `customer_id` está presente.
   - Pode ser relevante para gara

In [30]:
etapa_4 = executar_agente_langgraph(
    propositor,
    f"""
Produza uma proposta conceitual de correção para o incidente {incident_id}.

INVESTIGAÇÃO:
{etapa_2}

MÓDULOS AFETADOS:
{etapa_3}

Inclua plano de testes. Não gere patch executável ainda.
"""
)
print("ETAPA 4 — PROPOSTA")
print(etapa_4)

ETAPA 4 — PROPOSTA
Proposta conceitual de correção para o incidente INC-001:

**Contexto do problema:**
O erro `KeyError: 'customer_id'` ocorre porque a função `validar_pedido` acessa diretamente o campo `customer_id` (e outros campos obrigatórios) sem verificar se ele está presente no dicionário do pedido. Quando o campo está ausente (por exemplo, devido a schema drift do parceiro `marketplace_beta` que substituiu `customer_id` por `buyer_id`), ocorre exceção genérica que interrompe o pipeline.

---

### Proposta de mudança conceitual

1. **Validação robusta de campos obrigatórios:**
   - Modificar a função `validar_pedido` para verificar explicitamente a presença dos campos obrigatórios antes de acessá-los.
   - Em caso de ausência, lançar um erro estruturado (ex: `ValueError` com mensagem clara) que informe o campo ausente, o `order_id` e o parceiro (se disponível).
   - Isso evita `KeyError` genéricos e facilita o diagnóstico e tratamento.

2. **Tratamento de erros na ingestão:**
 

In [31]:
etapa_5 = executar_agente_langgraph(
    revisor,
    f"""
Revise a proposta para o incidente {incident_id}.

CLASSIFICAÇÃO:
{etapa_1}

INVESTIGAÇÃO:
{etapa_2}

MÓDULOS AFETADOS:
{etapa_3}

PROPOSTA:
{etapa_4}

Indique se a proposta pode seguir para preparação de patch na Aula 4 ou se deve ser bloqueada.
"""
)
print("ETAPA 5 — REVISÃO")
print(etapa_5)

ETAPA 5 — REVISÃO
Revisão da proposta para o incidente INC-001:

### Evidências confirmadas
- Logs do job `orders_daily_ingestion` mostram erro `KeyError: 'customer_id'` no pedido `MB-7781` do parceiro `marketplace_beta`.
- Histórico indica schema drift do parceiro `marketplace_beta` que substituiu `customer_id` por `buyer_id`.
- Código da função `validar_pedido` acessa diretamente campos obrigatórios, incluindo `customer_id`, sem verificar presença, o que pode causar `KeyError`.
- O pipeline de ingestão (`processar_lote_pedidos`) chama `validar_pedido` sem tratamento de exceções, o que aborta o processamento ao encontrar erro.
- A transformação assume que os pedidos já foram validados e contém os campos obrigatórios.
- Testes atuais cobrem casos de campo vazio, mas não ausência do campo `customer_id`.

### Avaliação da proposta

1. **Validação robusta de campos obrigatórios:**
   - A proposta de modificar `validar_pedido` para checar explicitamente a presença dos campos obrigatórios a

## Consolidação do resultado

A saída consolidada reúne as etapas em um único artefato JSON salvo em `outputs/reports/` para revisão. Pontos para revisar manualmente:
- Confirme que `classificacao`, `investigacao` e `modulos_afetados` contêm trechos/caminhos que de fato existem no repositório;
- Verifique se a `proposta_conceitual` inclui um plano de testes claro e reproduzível;
- Assegure que `revisao` documente riscos e bloqueios — o revisor deve explicitar se a proposta exige revisão humana antes de avançar.

Observação para instrutores: use o artefato consolidado como base para exercícios de revisão em sala (por exemplo, dividir turmas para validar evidências vs. avaliar o plano de testes).

In [32]:
resultado_langgraph = {
    "incident_id": incident_id,
    "classificacao": etapa_1,
    "investigacao": etapa_2,
    "modulos_afetados": etapa_3,
    "proposta_conceitual": etapa_4,
    "revisao": etapa_5,
    "observacao": "Nenhum arquivo foi alterado. A proposta prepara uma possível etapa de patch na Aula 4.",
}

caminho = salvar_json_saida("reports/incidente_inc001_langgraph_multiagentes.json", resultado_langgraph)
print("Arquivo salvo em:", caminho)

Arquivo salvo em: outputs/reports/incidente_inc001_langgraph_multiagentes.json


## O que observar

Compare este resultado com o agente único do notebook anterior e use as perguntas abaixo como checklist de revisão:

Checklist de validação:
- **Evidência**: cada agente citou caminhos/trechos verificáveis? (use `ler_arquivo` para confirmar);
- **Conexão**: os `modulos_afetados` apontam para arquivos que efetivamente contêm a lógica mencionada na investigação?;
- **Rastreabilidade**: é possível seguir a cadeia 
 nas saídas?;
- **Segurança**: o `revisor` identificou qualquer ação que deva ser bloqueada ou exigir revisão humana?;
- **Custo vs. benefício**: a decomposição gerou valor pedagógico e técnico suficiente para justificar a complexidade adicionada?

Sugestões de atividade em sala:
- Divida alunos em duplas: uma dupla valida evidências, outra propõe testes adicionais;
- Faça um exercício de 'ataque' onde um grupo tenta induzir um falso positivo e o outro valida via `ler_arquivo` e `executar_testes`.

## Orquestração com comando único (LangGraph)

Em vez de executar `etapa_1`, `etapa_2`, ... manualmente, podemos montar um grafo de estado com nós sequenciais (`classificador -> investigador -> mapeador -> propositor -> revisor`) e disparar tudo com um único `invoke`.

Vantagens desta abordagem:
- Pipeline reproduzível com uma única entrada (`incident_id`);
- Menos código operacional no notebook;
- Estrutura pronta para evoluir para fluxos condicionais (ex.: bloqueio antecipado).

In [33]:
from typing import TypedDict
from langgraph.graph import StateGraph, END


class EstadoInvestigacao(TypedDict, total=False):
    incident_id: str
    etapa_1: str
    etapa_2: str
    etapa_3: str
    etapa_4: str
    etapa_5: str


def no_classificador(state: EstadoInvestigacao):
    incident_id = state["incident_id"]
    etapa_1 = executar_agente_langgraph(
        classificador,
        f"Classifique o incidente {incident_id}. Consulte o ticket e indique quais evidências devem ser buscadas.",
    )
    return {"etapa_1": etapa_1}


def no_investigador(state: EstadoInvestigacao):
    incident_id = state["incident_id"]
    etapa_2 = executar_agente_langgraph(
        investigador,
        f"""
Investigue o incidente {incident_id} usando a classificação abaixo.

CLASSIFICAÇÃO:
{state['etapa_1']}

Formule hipóteses técnicas sustentadas por evidências consultadas via tools.
""",
    )
    return {"etapa_2": etapa_2}


def no_mapeador(state: EstadoInvestigacao):
    incident_id = state["incident_id"]
    etapa_3 = executar_agente_langgraph(
        mapeador,
        f"""
Mapeie módulos afetados no target_project para o incidente {incident_id}.

INVESTIGAÇÃO:
{state['etapa_2']}

Use tools para listar e ler arquivos relevantes do target_project.
""",
    )
    return {"etapa_3": etapa_3}


def no_propositor(state: EstadoInvestigacao):
    incident_id = state["incident_id"]
    etapa_4 = executar_agente_langgraph(
        propositor,
        f"""
Produza uma proposta conceitual de correção para o incidente {incident_id}.

INVESTIGAÇÃO:
{state['etapa_2']}

MÓDULOS AFETADOS:
{state['etapa_3']}

Inclua plano de testes. Não gere patch executável ainda.
""",
    )
    return {"etapa_4": etapa_4}


def no_revisor(state: EstadoInvestigacao):
    incident_id = state["incident_id"]
    etapa_5 = executar_agente_langgraph(
        revisor,
        f"""
Revise a proposta para o incidente {incident_id}.

CLASSIFICAÇÃO:
{state['etapa_1']}

INVESTIGAÇÃO:
{state['etapa_2']}

MÓDULOS AFETADOS:
{state['etapa_3']}

PROPOSTA:
{state['etapa_4']}

Indique se a proposta pode seguir para preparação de patch na Aula 4 ou se deve ser bloqueada.
""",
    )
    return {"etapa_5": etapa_5}


builder = StateGraph(EstadoInvestigacao)
builder.add_node("classificador", no_classificador)
builder.add_node("investigador", no_investigador)
builder.add_node("mapeador", no_mapeador)
builder.add_node("propositor", no_propositor)
builder.add_node("revisor", no_revisor)

builder.set_entry_point("classificador")
builder.add_edge("classificador", "investigador")
builder.add_edge("investigador", "mapeador")
builder.add_edge("mapeador", "propositor")
builder.add_edge("propositor", "revisor")
builder.add_edge("revisor", END)

pipeline_multiagente = builder.compile()

resultado_pipeline = pipeline_multiagente.invoke({"incident_id": "INC-001"})

print("ETAPA 1 — CLASSIFICAÇÃO")
print(resultado_pipeline.get("etapa_1", ""))
print("\nETAPA 2 — INVESTIGAÇÃO")
print(resultado_pipeline.get("etapa_2", ""))
print("\nETAPA 3 — MÓDULOS AFETADOS")
print(resultado_pipeline.get("etapa_3", ""))
print("\nETAPA 4 — PROPOSTA")
print(resultado_pipeline.get("etapa_4", ""))
print("\nETAPA 5 — REVISÃO")
print(resultado_pipeline.get("etapa_5", ""))

ETAPA 1 — CLASSIFICAÇÃO
Incidente INC-001 foi classificado com severidade inicial alta.

Resumo do incidente:
- Sistema afetado: mini_orders_pipeline
- Sintoma: Falha no job `orders_daily_ingestion` na etapa de ingestão de pedidos.
- Erro observado: `KeyError: 'customer_id'`
- Impacto: Pedidos do parceiro `marketplace_beta` não foram processados na janela esperada, afetando o dashboard diário de receita.

Evidências que devem ser consultadas para investigação:
1. Logs do job `orders_daily_ingestion` para identificar onde ocorre o `KeyError`.
2. Dados do lote enviado pelo parceiro `marketplace_beta` para verificar se o campo `customer_id` está presente e correto.
3. Código e testes relacionados à validação do schema dos pedidos, especialmente a função que valida o campo `customer_id`.
4. Histórico de alterações recentes no pipeline de ingestão e transformação de pedidos.
5. Runbooks ou procedimentos para tratamento de erros de ingestão.

Posso ajudar a buscar essas evidências específica

In [34]:
from IPython.display import Markdown, display

mermaid = pipeline_multiagente.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classificador(classificador)
	investigador(investigador)
	mapeador(mapeador)
	propositor(propositor)
	revisor(revisor)
	__end__([<p>__end__</p>]):::last
	__start__ --> classificador;
	classificador --> investigador;
	investigador --> mapeador;
	mapeador --> propositor;
	propositor --> revisor;
	revisor --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```